In [1]:
import numpy as np
import matplotlib.pyplot as plt
import animation
import sys
np.random.seed(0)

In [2]:
def construct_lap_dirichlet(nx):
    '''Construct laplacian operator with dirichlet BCs (zeros in corners)'''
    
    lap = -2*np.diag(np.ones(nx)) + 1*np.diag(np.ones(nx-1), 1) + 1*np.diag(np.ones(nx-1), -1)

    
    return lap

def construct_lap_periodic(nx):
    '''Construct laplacian operator with periodic BCs'''
    
    lap = -2*np.diag(np.ones(nx)) + 1*np.diag(np.ones(nx-1), 1) + 1*np.diag(np.ones(nx-1), -1)
    
    lap[0,-1] = 1    # Periodic BCs
    lap[-1,0] = 1 
    
    return lap

In [ ]:
# iteration params
dt = 0.00001
nt = 200000

dx = .10625*2.0 # near the boundary
nx = 100
x = np.linspace(0, dx*nx, nx)

# PDE params
r = 1.0
# qc = 3
qc = np.pi/(dx*nx) * 5

b3 = 0.1 
b5 = 0.1

kappa_e = 1.0
kappa_k = 6.0

beta_e = 5
d_e = 30

params = (r, qc, b3, b5, kappa_k, kappa_e, beta_e, d_e)
u0 = np.random.rand(nx) * .01 - .01/2
e0 = np.ones(nx) * 0.05

# u0 = u_stored[:,-1]
# e0 = e_stored[:,-1]

# integrate forward in time
def integrate_explicit(u0, e0, params, nx, dx, nt, dt, bc='dirichlet'):
    '''Integrate SH35 coupled to expander according to explicit Euler.'''
    
    r, qc, b3, b3, kappa_k, kappa_e, beta_e, d_e = params
    if bc=='dirichlet':
        lap_op = construct_lap_dirichlet(nx) / dx**2
    elif bc=='periodic':
        lap_op = construct_lap_periodic(nx) / dx**2
    qc_op = np.eye(nx) * qc
    
    # d = 1
    # lap_op_2 = lap_op * d # new diffusive operator
    
    # initialize fields
    u_stored = np.zeros((nx, nt))
    u_stored[:,0] = u0
    
    e_stored = np.zeros((nx, nt))
    e_stored[:,0] = e0
    
    # step forward in time
    for i in range(1, nt):
        
        if int(100*i/nt) > int(100*(i-1)/nt):
            sys.stdout.write("\rIteration {0}/{1} ({2}%)".format(i, nt, int(100*i/nt)))
            sys.stdout.flush()
            
        u = u_stored[:,i-1]
        e = e_stored[:,i-1]
        
        # update q_c due to coupling with E
        # reg = 0.01 # regularizer
        # qc = kappa_k * e
        # qc_op = np.eye(nx) * qc
        
        # update e
        # nonlin = kappa_e * np.abs(u)
        # e_stored[:,i] = e + dt*(nonlin - beta_e*e + d_e*lap_op@e) + np.random.normal(loc=0.0, scale=0.005*0, size=nx)
        
            
        # update u
        nonlin = b3*u**3 - b5*u**5
        harmonic_op = np.linalg.matrix_power(qc_op@qc_op + lap_op, 2) #+ lap_op_2
        
        u_stored[:,i] = u + dt*(r*u - harmonic_op@u + nonlin) + np.random.normal(loc=0.0, scale=.05*0, size=nx)
        if bc=='dirichlet': 
            u_stored[0,i] = 0; u_stored[-1,i] = 0  
            # e_stored[0,i] = 0; e_stored[-1,i] = 0  
    
        
    return u_stored#, e_stored
